In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")

### StateBackend

In [2]:
import os
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

StateBackend:

This is the default. Files live in LangGraph state for the current thread, so they persist across turns in the same conversation but are not shared across threads. It is best as a scratchpad for temporary plans, intermediate outputs, and large tool outputs that the agent may offload and read back later. One subtle point the docs mention: the state backend is shared between the supervisor and subagents, so files written by a subagent stay available afterward.

In [5]:
# -----------------------------------------------------------------------------
# 1. Create the agent — these two are equivalent
# -----------------------------------------------------------------------------
agent = create_deep_agent(model="ollama:glm-4.7:cloud")

# Under the hood this is what `agent` is doing — explicit StateBackend:
agent2 = create_deep_agent(
    model="ollama:glm-4.7:cloud",
    backend=StateBackend(),
)

In [6]:
# -----------------------------------------------------------------------------
# 2. Invoke the agent and ask it to WRITE a file
#    (StateBackend keeps that file inside LangGraph state)
# -----------------------------------------------------------------------------
result = agent2.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. create app system architecture\n2. Iterate over the architecture for improvement\n3. Create dependency of aws services\n"
            "Then tell me you've done it."
        )
    }]
})
# The agent's final natural-language reply
print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
Done. I've created /notes/todo.txt with the exact content you specified.


In [10]:
# the files /notes/todo.txt s created in the RAM for now, stored in Langgraph state, and can be retrieved like this:
print("\n--- Files in LangGraph state ------------------------------------")


--- Files in LangGraph state ------------------------------------


In [11]:
result

{'messages': [HumanMessage(content="Create a file at /notes/todo.txt with exactly this content:\n1. create app system architecture\n2. Iterate over the architecture for improvement\n3. Create dependency of aws services\nThen tell me you've done it.", additional_kwargs={}, response_metadata={}, id='f9db6f2f-7fb2-4f30-86bb-6814ad0dd214'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'glm-4.7', 'created_at': '2026-06-21T13:11:40.402539682Z', 'done': True, 'done_reason': 'stop', 'total_duration': 930422909, 'load_duration': None, 'prompt_eval_count': 6204, 'prompt_eval_duration': None, 'eval_count': 152, 'eval_duration': None, 'logprobs': None, 'model_name': 'glm-4.7', 'model_provider': 'ollama'}, id='lc_run--019eea4e-e3f2-7181-b5b6-7679e1ccb799-0', tool_calls=[{'name': 'write_file', 'args': {'file_path': '/notes/todo.txt', 'content': '1. create app system architecture\n2. Iterate over the architecture for improvement\n3. Create dependency of aws services'}, 'i

In [12]:
# -----------------------------------------------------------------------------
# 3. CHECK the backend is working
#    With StateBackend, written files appear under result["files"]
# -----------------------------------------------------------------------------
print("\n--- Backend check -----------------------------------------------")
files = result.get("files", {})

if files:
    print(f"✅ StateBackend is working — {len(files)} file(s) in state:")
    for path, content in files.items():
        print(f"\n📄 {path}\n{'-' * 40}\n{content}")
else:
    print("⚠️  No files found in state. Either the agent didn't write a file, "
          "or the backend isn't wired up correctly.")


--- Backend check -----------------------------------------------
✅ StateBackend is working — 1 file(s) in state:

📄 /notes/todo.txt
----------------------------------------
{'content': '1. create app system architecture\n2. Iterate over the architecture for improvement\n3. Create dependency of aws services', 'encoding': 'utf-8', 'created_at': '2026-06-21T13:11:40.407209+00:00', 'modified_at': '2026-06-21T13:11:40.407209+00:00'}


In [13]:
# -----------------------------------------------------------------------------
# 4. Prove persistence WITHIN the same thread:
#    feed the returned state back in and ask it to READ the file
# -----------------------------------------------------------------------------
followup = agent2.invoke({
    # carry forward prior messages + the files state
    "messages": result["messages"] + [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me ."
    }],
    "files": result.get("files", {}),   # <-- pass the virtual filesystem along
})

print("\n--- Read-back (same thread) -------------------------------------")
print(followup["messages"][-1].content)


--- Read-back (same thread) -------------------------------------
```
1. create app system architecture
2. Iterate over the architecture for improvement
3. Create dependency of aws services
```


### FileSystemBackend(local disk)


FilesystemBackend:


This maps agent file operations to real files on your machine or mounted storage under a root_dir. It is useful for local development, coding assistants, CI sandboxes, or persistent volumes. The big warning is that this is real disk access, so secrets and irreversible edits are in scope. The page strongly recommends virtual_mode=True; without it, even setting root_dir is not a real safety boundary. Another important recommendation: do not use FilesystemBackend by itself for most real projects, because Deep Agents also store internal artifacts like /large_tool_results/ and /conversation_history/. If you use only FilesystemBackend, those internal files end up mixed into your project directory.

In [14]:
# -----------------------------------------------------------------------------
# 1. Create the agent with a real-disk backend
#    root_dir="." -> files land relative to your current working directory
#    virtual_mode=True -> agent uses virtual paths like /notes/todo.txt,
#                         mapped onto root_dir
# -----------------------------------------------------------------------------
from deepagents.backends import FilesystemBackend
ROOT = "."

agent=create_deep_agent(model="ollama:glm-4.7:cloud",backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True))
print(f"✅ Agent created with FilesystemBackend(root_dir={ROOT!r}).")
print("   Files written by the agent will appear on your ACTUAL disk.")

✅ Agent created with FilesystemBackend(root_dir='.').
   Files written by the agent will appear on your ACTUAL disk.


In [15]:
# -----------------------------------------------------------------------------
# 2. Invoke the agent and ask it to WRITE a file
# -----------------------------------------------------------------------------
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. create app system architecture\n2. Iterate over the architecture for improvement\n3. Create dependency of aws services\n"
            "Then tell me you've done it."
        )
    }]
})

print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
Done. Created /notes/todo.txt with the specified content.


In [10]:
# -----------------------------------------------------------------------------
# 3. CHECK the backend is working — look on the REAL disk
#    With virtual_mode=True, /notes/todo.txt maps to ./notes/todo.txt
# -----------------------------------------------------------------------------
from pathlib import Path
print("\n--- Backend check (real filesystem) -----------------------------")
disk_path = Path(ROOT) / "notes" / "todo.txt"

if disk_path.exists():
    print(f"✅ FilesystemBackend is working — file exists on disk:")
    print(f"📄 {disk_path.resolve()}\n{'-' * 40}")
    print(disk_path.read_text())
else:
    print(f"⚠️  Expected file not found at {disk_path.resolve()}")
    print("    The agent may not have called the write tool, or the path "
          "mapping differs.")


--- Backend check (real filesystem) -----------------------------
✅ FilesystemBackend is working — file exists on disk:
📄 E:\agenticAI\deepagentscourse\deepagentsdemo\notes\todo.txt
----------------------------------------
1. Record video
2. Edit video
3. Upload video


In [16]:
# -----------------------------------------------------------------------------
# 4. Prove persistence ACROSS sessions:
#    Unlike StateBackend, this file survives even after Python exits.
#    A brand-new agent (fresh state) can read it back from disk.
# -----------------------------------------------------------------------------
fresh_agent = create_deep_agent(
    model="ollama:glm-4.7:cloud",
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

followup = fresh_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]
    # NOTE: no `files` state passed in — the file is read straight from disk
})

print("\n--- Read-back with a FRESH agent (proves disk persistence) ------")
print(followup["messages"][-1].content)


--- Read-back with a FRESH agent (proves disk persistence) ------
1. create app system architecture
2. Iterate over the architecture for improvement
3. Create dependency of aws services



Deep Agent — StoreBackend verification
--------------------------------------
Creates a deep agent backed by a LangGraph store, invokes it to write a
file on one thread, then proves the backend works by reading that file
back on a DIFFERENT thread — something StateBackend cannot do.
"""

StoreBackend


This stores files in a LangGraph BaseStore, which makes them durable across threads. This is the right choice for long-term memory, reusable instructions, or any cross-run persistence. For local development you can pass a store such as InMemoryStore, but on LangSmith Deployment you omit store because the platform provisions one automatically. The page emphasizes that for multi-user deployments you should always define a namespace so users or tenants do not share the same storage accidentally.

In [18]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store = InMemoryStore() #a file will be sttored in the RAM for now, stored in Langgraph state, and can be retrieved like we did above with StateBackend. The difference is that StoreBackend gives us more control and visibility into that state, and allows us to share it across agents and sessions.

agent = create_deep_agent(
    model="ollama:glm-4.7:cloud",
    backend=StoreBackend(
        # Local dev: static namespace. No deployment runtime needed.
        # In a LangSmith Deployment you'd use the user-identity version.
        namespace=lambda rt: ("demo-user",),
    ),
    store=store,
)

print("✅ Agent created with StoreBackend + static namespace.")

✅ Agent created with StoreBackend + static namespace.


In [19]:
import os
import uuid
# -----------------------------------------------------------------------------
# 2. THREAD 1 — write a file
# -----------------------------------------------------------------------------
thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Create a file at /notes/todo.txt with exactly this content:\n"
                "1. create app system architecture\n2. Iterate over the architecture for improvement\n3. Create dependency of aws services\n"
                "Then tell me you've done it."
            )
        }]
    },
    config=thread_1,
)

print("\n--- Agent reply (thread 1) --------------------------------------")
print(result["messages"][-1].content)


--- Agent reply (thread 1) --------------------------------------
Done. Created /notes/todo.txt with the specified content.


In [20]:
# --- Thread 2: read back on a DIFFERENT thread ---------------------------
thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}
followup = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]},
    config=thread_2,
)
print("\n--- Read-back on a different thread ---")
print(followup["messages"][-1].content)


--- Read-back on a different thread ---
1. create app system architecture
2. Iterate over the architecture for improvement
3. Create dependency of aws services


With StoreBackend + InMemoryStore, the file is not saved to disk at all. It lives in RAM, inside the InMemoryStore object, as an entry keyed under the namespace.

# Deep Agent Backends — Where Your Files Actually Live

Every Deep Agent works with a **virtual filesystem**: its tools read and write
files using paths like `/notes/todo.txt`. But those paths are an abstraction.
The **backend** decides where that data physically lives — and that single
choice changes everything about persistence, sharing, and durability.

The agent code stays identical across all three. Only the backend changes.

---

## 1. StateBackend (default)

Files live **in the agent's LangGraph state** — i.e. in RAM, tied to a single
thread. They're available during the run via `result["files"]`, and you must
manually carry that state forward to keep using them.

- **Where:** in-memory dict, inside one thread's state
- **Survives:** within a single conversation/thread only
- **Gone when:** the thread ends or the process exits
- **Use it for:** ephemeral scratch space, temporary working files

## 2. FilesystemBackend

Files are written to your **real disk**, under `root_dir`. With
`virtual_mode=True`, a virtual path like `/notes/todo.txt` maps to
`root_dir/notes/todo.txt`. These are genuine files you can open in an editor
or `cat` from a terminal.

- **Where:** your actual filesystem, relative to `root_dir`
- **Survives:** across process restarts — it's a real file
- **Use it for:** when the agent should edit real project files
- **⚠️ Caution:** grants the agent real read/write disk access — point it at a
  scratch directory, not anything important

## 3. StoreBackend

Files live in a **LangGraph store**, scoped by a `namespace`. This is the only
backend whose files persist **across threads/conversations** — write on one
thread, read back on another. With `InMemoryStore` this works within a single
process; swap in a persistent store (e.g. Postgres-backed) for true
cross-restart durability.

- **Where:** inside the store object, under a namespace key
- **Survives:** across threads sharing the same store
- **Gone when:** the process exits (if using `InMemoryStore`) — use a
  persistent store to outlive restarts
- **Use it for:** long-term memory, per-user file spaces

---

### Quick comparison

| Backend            | Lives in        | Cross-thread? | Survives restart?      | Real file on disk? |
|--------------------|-----------------|:-------------:|:----------------------:|:------------------:|
| `StateBackend`     | LangGraph state |       ❌      |           ❌           |         ❌         |
| `FilesystemBackend`| Your disk       |       ✅      |           ✅           |         ✅         |
| `StoreBackend`     | A LangGraph store |     ✅      | only w/ persistent store |       ❌         |

> **Mental model:** the agent never knows the difference. Its tools always
> speak in file paths — the backend quietly translates every read/write into
> the right operation underneath. That's what lets you swap backends without
> touching a line of agent logic.

LocalShellBackend


This is essentially FilesystemBackend plus execute, meaning the agent can run shell commands directly on the host machine. It is for trusted local development only. The docs are very explicit that this is not sandboxed, and virtual_mode=True does not make it safe because shell commands can still access arbitrary paths on the system. Use this only when you trust the agent and the environment is controlled.

ContextHubBackend


This stores the agent filesystem in a LangSmith Context Hub repo. It is basically a durable filesystem backed by Hub commits, not by a LangGraph store. Reads are served from a cache after the initial pull, and writes/edits become Hub commits. This is attractive if you want LangSmith-native persistence and commit history. It also has a specific repo model: the agent repo mounts at the filesystem root, and linked skill repos appear under /skills/. That separation is intentional so skills can be shared and versioned independently.

CompositeBackend

This is the router backend. It lets different path prefixes map to different backends. This is the most flexible pattern and, practically, the one the docs keep pointing toward. A common setup is:

default backend: StateBackend()

/workspace/: FilesystemBackend(...)

/memories/: StoreBackend(...)

That way the agent gets temporary scratch space, real project file access where you explicitly allow it, and durable long-term memory where you want it.

How to choose one

Here is the simplest decision rule:

Use StateBackend if you want safe thread-scoped scratch storage.


Use FilesystemBackend if the agent must work with real local files.


Use LocalShellBackend only if the agent must also run host shell commands and you trust the environment.


Use StoreBackend if memory must persist across threads or sessions.


Use ContextHubBackend if you want durable file persistence tied to LangSmith Hub repos and commit history.


Use CompositeBackend if you want a production-worthy mix of the above.